In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
# from google.colab import files
# files.upload()


In [2]:
def combine_prediction(_df, clf1=False, clf2=False, reg1=False, reg2=False):
    _df["date"] = pd.to_datetime(_df["ID"].apply(lambda s:s.split("_")[2]),infer_datetime_format=True)
    _df["s"] = _df["ID"].apply(lambda s:s.split("_")[0])
    _df["ss"] = _df["ID"].apply(lambda s:s.split("_")[1])
    _df["sku"] = _df["s"]+"_"+_df["ss"]
    _df.drop(["s","ss"],axis=1, inplace=True)
    _df = _df.sort_values(by=["sku","date"])

    _df["2w_lag"] = _df.groupby(["sku"])["Target_purchase_next_2w"].shift(1)
    _df["1w_fwd"] = _df.groupby(["sku"])["Target_purchase_next_1w"].shift(-1)
    _df.loc[_df["2w_lag"].isnull(),"2w_lag"] = _df.loc[_df["2w_lag"].isnull(),"Target_purchase_next_1w"]
    _df.loc[_df["1w_fwd"].isnull(),"1w_fwd"] = _df.loc[_df["1w_fwd"].isnull(),"Target_purchase_next_2w"]
    if clf1:
        print("clf 1w")
        _df["Target_purchase_next_1w"] = 0.5 * _df["Target_purchase_next_1w"] + 0.5 * _df["2w_lag"]
    if clf2:
        print("clf 2w")
        _df["Target_purchase_next_2w"] = 0.5 * _df["Target_purchase_next_2w"] + 0.5 * _df["1w_fwd"]
    _df.drop(["2w_lag","1w_fwd"],axis=1, inplace=True)


    _df["2w_lag"] = _df.groupby(["sku"])["Target_qty_next_2w"].shift(1)
    _df["1w_fwd"] = _df.groupby(["sku"])["Target_qty_next_1w"].shift(-1)
    _df.loc[_df["2w_lag"].isnull(),"2w_lag"] = _df.loc[_df["2w_lag"].isnull(),"Target_qty_next_1w"]
    _df.loc[_df["1w_fwd"].isnull(),"1w_fwd"] = _df.loc[_df["1w_fwd"].isnull(),"Target_qty_next_2w"]
    if reg1:
        print("reg 1w")
        _df["Target_qty_next_1w"] = 0.5 * _df["Target_qty_next_1w"] + 0.5 * _df["2w_lag"]
    if reg2:
        print("reg 2w")
        _df["Target_qty_next_2w"] = 0.5 * _df["Target_qty_next_2w"] + 0.5 * _df["1w_fwd"]
    _df.drop(["2w_lag","1w_fwd"],axis=1, inplace=True)
    #
    return _df

In [3]:
def load_submission(person,name):
    return pd.read_csv(f"./probe/subs/raw_exps/{person}/{name}.csv")

In [4]:
#{"name":"submission_9797", "clf1":0.968183436,"clf2":0.966706956,"reg1": 0.428002353,"reg2":0.701630427},

In [5]:
data  = [
    {"name":"submission_9762", "ord":1, "clf1":0.960058249,"clf2":0.955232982,"reg1":0.286893874,"reg2":0.461325135},
    {"name":"submission_9767", "ord":2, "clf1":0.957032784,"clf2":0.960963304,"reg1":0.320573591,"reg2":	0.471975589},
    {"name":"submission_9772", "ord":3, "clf1":0.964219385,"clf2":0.959493267,"reg1":0.430061311,"reg2":	0.597904004},
    {"name":"submission_9741", "ord":4, "clf1":0.963656342,"clf2":0.959060459,"reg1":0.79202222,"reg2":1.05219464},
    {"name":"submission_9783", "ord":5, "clf1":0.964219385,"clf2":0.959493267,"reg1":0.277733411,"reg2":0.453677481},

    {"name":"submission_9811", "ord":6,"clf1":0.968183436,"clf2":0.966706956,"reg1":0.277733411,"reg2":0.453677481},
    #{"name":"submission_9737", "clf1":,"clf2":,"reg1":,"reg2":},
]
pd.DataFrame(data)

,name,ord,clf1,clf2,reg1,reg2
0,submission_9762,1,0.960058,0.955233,0.286894,0.461325
1,submission_9767,2,0.957033,0.960963,0.320574,0.471976
2,submission_9772,3,0.964219,0.959493,0.430061,0.597904
3,submission_9741,4,0.963656,0.959060,0.792022,1.052195
4,submission_9783,5,0.964219,0.959493,0.277733,0.453677
5,submission_9811,6,0.968183,0.966707,0.277733,0.453677


In [6]:
%%time
df1 = pd.read_csv("/content/cat_balanced.csv")
df1 = combine_prediction(df1,  clf1=True, reg1=False)
df1 = df1.set_index("ID")

df2 = pd.read_csv("/content/lgb_balanced.csv")
df2 = combine_prediction(df2,  clf1=True, reg1=False)
df2 = df2.set_index("ID")

df3 = pd.read_csv("/content/ens_balanced.csv")
df3 = combine_prediction(df3,  clf1=True, reg1=False)
df3 = df3.set_index("ID")



clf 1w
clf 1w
clf 1w
CPU times: user 4.63 s, sys: 258 ms, total: 4.88 s
Wall time: 8.12 s


In [7]:
df2.head()

,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w,date,sku
ID,,,,,,
339_1_20250922,0.003681,0.0,0.003681,0.0,2025-09-22,339_1
339_1_20250929,0.003681,0.0,0.003681,0.0,2025-09-29,339_1
339_1_20251006,0.003681,0.0,0.003681,0.0,2025-10-06,339_1
339_1_20251013,0.003681,0.0,0.003681,0.0,2025-10-13,339_1
339_1_20251020,0.003681,0.0,0.003681,0.0,2025-10-20,339_1


In [8]:
pd.DataFrame(data)

,name,ord,clf1,clf2,reg1,reg2
0,submission_9762,1,0.960058,0.955233,0.286894,0.461325
1,submission_9767,2,0.957033,0.960963,0.320574,0.471976
2,submission_9772,3,0.964219,0.959493,0.430061,0.597904
3,submission_9741,4,0.963656,0.959060,0.792022,1.052195
4,submission_9783,5,0.964219,0.959493,0.277733,0.453677
5,submission_9811,6,0.968183,0.966707,0.277733,0.453677


In [9]:
df = df1.copy()

c1 = "Target_purchase_next_1w"
df[c1] = df1[c1]


c2 = "Target_purchase_next_2w"
df[c2] =  df1[c2] #1

r1 = "Target_qty_next_1w"
df[r1] = df2[r1]

r2 = "Target_qty_next_2w"
df[r2] = df2[r2]

In [10]:
df

,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w,date,sku
ID,,,,,,
339_1_20250922,0.004747,0.0,0.008929,0.0,2025-09-22,339_1
339_1_20250929,0.006838,0.0,0.008929,0.0,2025-09-29,339_1
339_1_20251006,0.006970,0.0,0.009424,0.0,2025-10-06,339_1
339_1_20251013,0.007218,0.0,0.009424,0.0,2025-10-13,339_1
339_1_20251020,0.007451,0.0,0.010187,0.0,2025-10-20,339_1
...,...,...,...,...,...,...
782_99_20250929,0.000392,0.0,0.000593,0.0,2025-09-29,782_99
782_99_20251006,0.025277,0.0,0.255798,0.0,2025-10-06,782_99
782_99_20251013,0.357405,0.0,0.666650,0.0,2025-10-13,782_99


In [11]:
df[["Target_purchase_next_1w","Target_qty_next_1w","Target_purchase_next_2w","Target_qty_next_2w"]].to_csv("submission.csv",index=True)